# Inventory Replenishment Forecasting

## Project Overview

This project forecasts **daily inventory consumption** to determine optimal replenishment
timing and quantities. We generate synthetic daily consumption data for a warehouse item
over 2 years, then forecast the next 14 days to calculate reorder points and safety stock.

**Models**: Naive baselines, LazyPredict (tabular), FLAML AutoML, StatsForecast.

**Forecast horizon**: 14 days — aligned with typical supplier lead times.

## Learning Objectives

1. Connect demand forecasting to inventory management decisions
2. Calculate reorder points and safety stock from forecasts
3. Build and compare naive, ML, and statistical forecasting models
4. Use FLAML and StatsForecast for automated demand prediction
5. Understand the cost of forecast errors in inventory context

## Problem Statement

A warehouse needs to know when to reorder a key item and how much to order.
Given historical daily consumption, forecast the next 14 days of demand to
determine if current stock will last until the next delivery.

## Why This Project Matters

Poor inventory forecasting leads to either stockouts (lost sales, production halts)
or excess inventory (tied-up capital, spoilage). Forecasting-driven replenishment
optimizes the balance between service level and carrying cost.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily item consumption
- Stable base demand with weekly seasonality
- Gradual upward trend (growing business)
- Occasional demand spikes (promotions)
- Random noise

| Column | Description |
|--------|-------------|
| `date` | Date |
| `consumption` | Daily units consumed |

## Dataset Source and License Notes

Synthetically generated for educational purposes.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_DAYS = 730
HORIZON = 14
VAL_DAYS = 14
TEST_DAYS = 14
SEASON_LENGTH = 7
LEAD_TIME_DAYS = 14   # supplier lead time
SERVICE_LEVEL = 0.95  # target service level
np.random.seed(SEED)
print(f"Config: {N_DAYS} days, horizon={HORIZON}, lead_time={LEAD_TIME_DAYS}d")

Config: 730 days, horizon=14, lead_time=14d


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_DAYS, freq='D')
t = np.arange(N_DAYS)

base = 150 + 0.1 * t  # gradual growth
weekly = 20 * np.sin(2 * np.pi * t / 7)  # weekly cycle

# Occasional spikes (promotions ~every 60 days)
spikes = np.zeros(N_DAYS)
for i in range(0, N_DAYS, 60):
    spike_start = min(i + np.random.randint(0, 30), N_DAYS - 3)
    for j in range(min(3, N_DAYS - spike_start)):
        spikes[spike_start + j] = 80

noise = np.random.normal(0, 15, N_DAYS)
consumption = base + weekly + spikes + noise
consumption = np.maximum(consumption, 20).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'consumption': consumption})
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

Dataset shape: (730, 2)
Date range: 2022-01-01 00:00:00 to 2023-12-31 00:00:00


,date,consumption
0,2022-01-01,120
1,2022-01-02,158
2,2022-01-03,176
3,2022-01-04,145
4,2022-01-05,143


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0
assert (df['consumption'] > 0).all()
assert df['date'].is_monotonic_increasing
assert len(df) == N_DAYS
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(df['date'], df['consumption'], linewidth=0.6)
axes[0, 0].set_title('Daily Consumption Over Time')

axes[0, 1].hist(df['consumption'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Consumption Distribution')

# Day-of-week pattern
df['dow'] = df['date'].dt.dayofweek
dow_avg = df.groupby('dow')['consumption'].mean()
axes[1, 0].bar(dow_avg.index, dow_avg.values, alpha=0.7)
axes[1, 0].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
axes[1, 0].set_title('Avg Consumption by Day of Week')

# Rolling 7-day average
df['rolling_7d'] = df['consumption'].rolling(7).mean()
axes[1, 1].plot(df['date'], df['rolling_7d'], linewidth=1)
axes[1, 1].set_title('7-Day Rolling Average')

plt.tight_layout()
plt.savefig('eda_inventory.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA plots saved.")

EDA plots saved.


## Target Analysis

In [7]:
print("Consumption Statistics:")
print(df['consumption'].describe())
print(f"\nAvg daily consumption: {df['consumption'].mean():.1f} units")
print(f"Max daily consumption: {df['consumption'].max()} units")
print(f"CV: {df['consumption'].std() / df['consumption'].mean():.3f}")

Consumption Statistics:
count    730.000000
mean     190.768493
std       34.505757
min      108.000000
25%      165.000000
50%      189.000000
75%      213.000000
max      317.000000
Name: consumption, dtype: float64

Avg daily consumption: 190.8 units
Max daily consumption: 317 units
CV: 0.181


## Train / Validation / Test Split Strategy

Time-aware split:
- Train: first 702 days
- Validation: 14 days
- Test: last 14 days

In [8]:
train = df.iloc[:-(VAL_DAYS + TEST_DAYS)].copy()
val = df.iloc[-(VAL_DAYS + TEST_DAYS):-TEST_DAYS].copy()
test = df.iloc[-TEST_DAYS:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — daily integer consumption. Tree-based models handle raw features.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data, target='consumption'):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[target].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[target].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[target].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', 'consumption', 'rolling_7d']]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:8.2f} | RMSE: {rmse:8.2f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_DAYS, train['consumption'].iloc[-1])
results.append(calc_metrics(test['consumption'].values, naive_pred, "Naive (Last Value)"))

src = df_full['consumption'].values
ti = len(df_full) - TEST_DAYS
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_DAYS]
results.append(calc_metrics(test['consumption'].values, sn_pred, "Seasonal Naive (7d)"))

Naive (Last Value)             | MAE:    27.79 | RMSE:    34.50 | MAPE: 10.77%
Seasonal Naive (7d)            | MAE:    27.57 | RMSE:    41.42 | MAPE: 10.16%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_DAYS
cv = ct - VAL_DAYS
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv]['consumption']
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct]['consumption']

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                           Adjusted R-Squared  R-Squared        RMSE  Time Taken
Model                                                                           
KernelRidge                        969.821548 -73.524734  187.289069    0.019001
GaussianProcessRegressor           611.204627 -45.938817  148.637463    0.045166
MLPRegressor                       181.771782 -12.905522   80.901366    0.304503
QuantileRegressor                   43.173306  -2.244100   39.075933    0.067071
DummyRegressor                      40.779339  -2.059949   37.950659    0.006845
DecisionTreeRegressor               30.716822  -1.285909   32.801350    0.012254
NuSVR                               20.690064  -0.514620   26.700174    0.023164
SVR                                 19.914625  -0.454971   26.169137    0.023913
RANSACRegressor                     15.206151  -0.092781   22.679253    0.066700
OrthogonalMatchingPursuit           14.076715  -0.005901   21.759047    0.007195


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:]['consumption']
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: lgbm
FLAML (lgbm)                   | MAE:    14.66 | RMSE:    18.02 | MAPE:  6.62%
FLAML Test (lgbm)              | MAE:    28.27 | RMSE:    37.24 | MAPE: 10.57%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', 'consumption']].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'item1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_DAYS]

sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH), SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq='D', n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_DAYS).reset_index()

y_test_sf = test['consumption'].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))

print("StatsForecast complete.")

SF AutoARIMA                   | MAE:    24.99 | RMSE:    38.30 | MAPE:  8.98%
SF AutoETS                     | MAE:    31.01 | RMSE:    42.89 | MAPE: 11.49%
SF SeasonalNaive               | MAE:    31.21 | RMSE:    44.73 | MAPE: 11.64%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
              model       MAE      RMSE      MAPE
       FLAML (lgbm) 14.656510 18.022101  6.624437
       SF AutoARIMA 24.987041 38.299203  8.984204
Seasonal Naive (7d) 27.571429 41.416008 10.159893
 Naive (Last Value) 27.785714 34.503623 10.768915
  FLAML Test (lgbm) 28.270724 37.243497 10.570729
         SF AutoETS 31.009905 42.887765 11.485540
   SF SeasonalNaive 31.214285 44.730144 11.641384

Top 3:
              model       MAE      RMSE      MAPE
       FLAML (lgbm) 14.656510 18.022101  6.624437
       SF AutoARIMA 24.987041 38.299203  8.984204
Seasonal Naive (7d) 27.571429 41.416008 10.159893


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test['consumption'].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Period')
ax.set_ylabel('Daily Consumption')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

# Inventory replenishment calculation
print("\n--- Inventory Replenishment Analysis ---")
forecast_14d = flaml_test
total_demand_14d = forecast_14d.sum()
avg_daily = forecast_14d.mean()
std_daily = forecast_14d.std()

# Safety stock = z * sigma * sqrt(lead_time)
from scipy.stats import norm
z = norm.ppf(SERVICE_LEVEL)
safety_stock = z * std_daily * np.sqrt(LEAD_TIME_DAYS)
reorder_point = avg_daily * LEAD_TIME_DAYS + safety_stock

print(f"Forecasted 14-day demand: {total_demand_14d:.0f} units")
print(f"Avg daily demand: {avg_daily:.1f} units")
print(f"Safety stock ({SERVICE_LEVEL:.0%} SL): {safety_stock:.0f} units")
print(f"Reorder point: {reorder_point:.0f} units")


--- Inventory Replenishment Analysis ---
Forecasted 14-day demand: 3115 units
Avg daily demand: 222.5 units
Safety stock (95% SL): 88 units
Reorder point: 3203 units


## Error Analysis

In [17]:
errors = test['consumption'].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 22.73, Std: 29.50


## Interpretation and Business Insight

- Forecasts directly drive reorder point and safety stock calculations
- Higher forecast uncertainty → higher safety stock → higher carrying cost
- The best model balances accuracy with stability (low variance in errors)
- Weekly consumption patterns should inform delivery scheduling

## Limitations

1. Synthetic data — real inventory has supplier delays, quality issues, returns
2. Single-item analysis — real warehouses manage thousands of SKUs
3. No stockout censoring — actual consumption may be limited by available stock
4. Point forecast only — probabilistic forecasts would better inform safety stock

## How to Improve This Project

1. Add multi-SKU forecasting with ABC classification
2. Implement probabilistic forecasts for better safety stock
3. Add supplier lead time variability
4. Model promotion effects explicitly
5. Use Chronos-Bolt for zero-shot demand forecasting

## Production Considerations

- Daily automated retraining and reorder point updates
- Dashboard showing days-of-stock remaining per SKU
- Alert when predicted demand exceeds available stock within lead time
- Integration with purchase order systems

## Common Mistakes

1. Ignoring lead time variability in safety stock calculations
2. Using point forecasts without uncertainty for inventory decisions
3. Not accounting for demand spikes (promotions) in forecasts
4. Setting static reorder points instead of dynamic forecast-driven ones

## Mini Challenge / Exercises

1. Add a second item with intermittent demand — how does the approach change?
2. Vary the service level (90%, 95%, 99%) and observe safety stock changes
3. Simulate a stockout scenario and measure the business impact
4. Implement an (s, S) inventory policy using the forecast

## Final Summary / Key Takeaways

1. Inventory replenishment forecasting connects demand prediction to business decisions
2. Forecast accuracy directly impacts safety stock and carrying costs
3. Reorder point = expected demand during lead time + safety stock
4. Weekly seasonality patterns inform delivery scheduling
5. Probabilistic forecasts are preferred over point forecasts for inventory decisions

In [18]:
import json
metrics = {
    'project': 'Inventory Replenishment Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'reorder_point': float(reorder_point),
    'safety_stock': float(safety_stock),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Inventory Replenishment Forecasting — Complete ===")

Metrics saved.

=== Inventory Replenishment Forecasting — Complete ===
